# AI Coding Assistant

Help you with writing code comments, unit tests, code fixing, and **converting Python to C++**.



In [ ]:

import os
import anthropic
import gradio as gr
from dotenv import load_dotenv

load_dotenv(override=True)


MODEL      = "claude-sonnet-4-20250514"
MAX_TOKENS = 4096

SYSTEM_PROMPT = (
    "You are a senior software engineer fluent in both Python and C++. "
    "When converting to C++, produce clean, compilable C++17 code with proper "
    "memory management and idiomatic style."
)

print("✓ Imports loaded")

✓ Imports loaded


In [ ]:

def get_client():
    key = os.environ.get("ANTHROPIC_API_KEY", "")
    if not key:
        raise ValueError(
            "ANTHROPIC_API_KEY not set. "
            "Add it to your .env file or set it as an environment variable."
        )
    return anthropic.Anthropic(api_key=key)

print("✓ Client helper ready")

✓ Client helper ready


In [ ]:

TASK_OPTIONS = [
    "Add Comments",
    "Explain Code",
    "Write Unit Tests",
    "Fix Code",
    "Convert to C++",
]


def build_prompt(tasks, code):
    instructions = []

    if "Add Comments" in tasks:
        instructions.append("Add clear, helpful inline comments to the code.")
    if "Explain Code" in tasks:
        instructions.append("Explain what the code does in plain English.")
    if "Write Unit Tests" in tasks:
        instructions.append("Write comprehensive unit tests using pytest.")
    if "Fix Code" in tasks:
        instructions.append("Identify and fix any bugs or issues in the code.")
    if "Convert to C++" in tasks:
        instructions.append(
            "Convert the Python code to idiomatic, modern C++17. "
            "Include all necessary headers, use appropriate STL containers and algorithms, "
            "add a main() function with a usage example, and add comments explaining "
            "key differences from the Python version."
        )

    return f"Tasks:\n- {'\n- '.join(instructions)}\n\nPython Code:\n```python\n{code}\n```"


print("✓ Task builder ready")

✓ Task builder ready


In [ ]:

def run_code_assistant(model, tasks, code):
    if not code.strip():
        return "Please provide some Python code."
    if not tasks:
        return "Please select at least one task."

    try:
        client = get_client()
        user_prompt = build_prompt(tasks, code)

        response = client.messages.create(
            model=model,
            max_tokens=MAX_TOKENS,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_prompt}],
        )
        return response.content[0].text

    except anthropic.AuthenticationError:
        return "⚠ Invalid API key. Check your ANTHROPIC_API_KEY."
    except Exception as e:
        return f"⚠ Error: {e}"


print("✓ Generation function ready")

✓ Generation function ready


In [ ]:

SAMPLE_PYTHON = """\
class Stack:
    def __init__(self):
        self.items = []

    def push(self, item):
        self.items.append(item)

    def pop(self):
        if self.is_empty():
            raise IndexError("pop from empty stack")
        return self.items.pop()

    def peek(self):
        if self.is_empty():
            raise IndexError("peek at empty stack")
        return self.items[-1]

    def is_empty(self):
        return len(self.items) == 0

    def size(self):
        return len(self.items)


def is_balanced(expression):
    stack = Stack()
    pairs = {")": "(", "]": "[", "}": "{"}
    for char in expression:
        if char in "([{":
            stack.push(char)
        elif char in ")]}":
            if stack.is_empty() or stack.pop() != pairs[char]:
                return False
    return stack.is_empty()
"""

In [14]:

CLAUDE_MODELS = [
    "claude-sonnet-4-20250514",
    "claude-opus-4-20250514",
    "claude-haiku-4-5-20251001",
]

with gr.Blocks(title="AI Coding Assistant") as ui:

    gr.Markdown("# AI Coding Assistant")
    gr.Markdown(
        "Add comments, explain, fix, write tests, or **convert Python → C++**. "
        "Pick a Claude model, tick your tasks, paste code, and hit **Run**."
    )


    model = gr.Dropdown(
        choices=CLAUDE_MODELS,
        value=CLAUDE_MODELS[0],
        label="Claude Model",
    )


    tasks = gr.CheckboxGroup(
        choices=TASK_OPTIONS,
        value=["Convert to C++"],
        label="Tasks",
    )

    
    code_input = gr.Code(
        value=SAMPLE_PYTHON,
        language="python",
        label="Python Code",
        lines=25,
    )

    with gr.Row():
        run_btn   = gr.Button("▶ Run", variant="primary")
        clear_btn = gr.Button("🗑 Clear output")

    output = gr.Markdown(label="Output")

    run_btn.click(
        run_code_assistant,
        inputs=[model, tasks, code_input],
        outputs=output,
    )
    clear_btn.click(lambda: "", outputs=output)

ui.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
